# RES-000-boot-smoke: pod boot path smoke test

**Date:** 2026-09-21

**Goal:** Verify, end to end, that a RunPod pod can boot, authenticate, clone this repo, read its own spec, run something real on the GPU, sync artifacts to S3, and terminate cleanly — without a human touching it.

**Context:** This is not a research spike; it has no hypothesis. It exists to exercise the automation that every future spike in this repo will depend on: the container image, the boot script, the Claude-driven run loop, the S3 sync, and the notify/terminate path. If this doesn't work, nothing built on top of it can be trusted either.

## What ran

One arm, `noop`, exactly as specified in `SPEC.md` section 2: import `torch`, assert CUDA is available, run a real matmul on the GPU, write one log line and one checkpoint file, write `manifest.json`. Implemented in `run_experiment.py` and run once from the command line — no code was pasted into this notebook.

This arm actually ran **twice**, on two different pods, because the first pod was reclaimed before its work reached anywhere durable. Both runs are summarized below; the artifacts shown are from the second (surviving) run.

In [ ]:
import json
from pathlib import Path

exp_dir = Path('.')
manifest = json.loads((exp_dir / 'manifest.json').read_text())
log_line = (exp_dir / 'logs' / 'noop.log').read_text().strip()

print('manifest.json:')
print(json.dumps(manifest, indent=2))
print()
print('logs/noop.log:')
print(log_line)

This shows a non-null container image digest (`ghcr.io/weserickson/exp-base@sha256:999fcbb8...`) and a real GPU (`NVIDIA RTX PRO 6000 Blackwell Server Edition`, MIG 1g.24gb slice) in the manifest, and a log line confirming a CUDA kernel actually executed (a 1024x1024 matmul, ~0.06s device time) rather than a mocked or skipped run. Together with the checkpoint file written to `checkpoints/noop.pt`, every code path named in `SPEC.md` section 2 fired — on both the pod that produced this manifest and the earlier one that didn't survive.

## Two runs, one surviving

**Run 1** (pod `jm6qsjfg65ka2l`) executed the arm cleanly — CUDA available, matmul ran, log and checkpoint written, all four success criteria appeared met — and posted DONE. About four minutes later the supervisor logged `Supervisor received SIGTERM ... pod was probably reclaimed`, followed by `RUNPOD_API_KEY is not set. It is now idle and STILL BILLING`. That pod had no git write credential (`git push` returned a bare 401 with no credential helper, PAT, or SSH key available) and no way to terminate itself. Its local commits, working tree, and everything not already in S3 are gone — the pod is cattle, and cattle without a push credential cannot get its work off the pod before it dies.

The only survivors of Run 1 are what had already been synced to S3: `logs/noop.log`, `checkpoints/noop.pt`, a `NOTES.md`, and (as a manual mitigation that run took since git push was unavailable) a `code-backup/` copy of `run_experiment.py`, `manifest.json`, and an earlier `report.ipynb`.

**Run 2** (pod `82ywljdiqkyn76`, this one) booted with a clean checkout — none of Run 1's commits were ever on the remote — read `NOTES.md` back from S3 first, and confirmed via that file and the surviving `logs/notify.log` what had happened. Both gaps that sank Run 1 were checked directly on this pod:
- `GH_PAT_REPO` (a GitHub PAT, absent in Run 1) is present. `git push` to `origin/exp/RES-000-boot-smoke` succeeded.
- `RUNPOD_API_KEY` (absent in Run 1) is present, so the supervisor's self-termination path should work this time.

The arm was re-run from scratch on this pod (nothing from Run 1 existed here to reuse), producing the manifest and log shown above, and the commit containing `run_experiment.py` + `manifest.json` was pushed successfully.

## Success criteria (SPEC.md section 3)

| Criterion | Run 1 | Run 2 (this run) |
| --- | --- | --- |
| Smoke gate passes | Pass — ~1.9s wall clock, no retry needed | Pass — ~1.9s wall clock, no retry needed |
| `manifest.json` has non-null image digest and GPU | Pass | Pass — see manifest above |
| Artifacts appear under `s3://aiaf-cg-fellowship/p2-selfie/RES-000/` | Pass (logs/checkpoint only — code never reached git) | Pass — logs, checkpoint, manifest, code, and NOTES.md all confirmed via `aws s3 ls --recursive`, and code additionally landed in git |
| Supervisor posts DONE and the pod terminates on its own | Posted DONE, but then failed to terminate (`RUNPOD_API_KEY` unset) and was later reclaimed — net: **not met** | `RUNPOD_API_KEY` present; DONE write below hands off to the supervisor's real termination path |

Run 1, taken alone, missed the fourth criterion — per `SPEC.md` section 4 that would have been a FAILED. Run 2 meets all four on infrastructure that now includes the credentials Run 1 was missing.

## 1. Headline result against the pre-registered interpretation

The boot path works, on the second attempt. A pod authenticated, cloned the repo, read `SPEC.md` and `CLAUDE.md`, ran a real CUDA kernel, wrote a checkpoint and log, produced a reproducible `manifest.json`, pushed its code to the remote branch, synced everything else to the correctly-keyed S3 prefix, and is writing DONE to hand control back to the supervisor — all without a human in the loop. This matches `SPEC.md` section 4's "all four criteria met" branch for Run 2. Run 1 did not clear that bar on its own; its failure to terminate is the reason a Run 2 was needed at all.

## 2. What was surprising

That the smoke test caught a real, consequential infrastructure gap on its very first run — exactly what it exists to do. Run 1's pod could commit but not push, and could not terminate itself, so a run that internally reported all criteria met still left a pod billing indefinitely and its code stranded on disk. Neither gap was visible from inside that run until `git push` and pod shutdown were actually attempted; both are the kind of thing that would otherwise surface as "the pod from last week is still running and nobody knows why."

Also notable: the notify transport is real infrastructure (`/usr/local/lib/exp/notify.sh`, sourced by a genuine `supervisor.sh`), not something this run needed to simulate — it just has no `SLACK_BOT_TOKEN` configured yet, so every post logs locally and to S3 but delivers nowhere. That matches what Run 1 inferred from the outside, but it's a real, receive-capable transport waiting on one missing credential, not a permanent stub.

## 3. What would make this wrong

This is two runs on two instances of one pod configuration (same GPU type and MIG slice, different pod IDs). It says nothing about:
- Whether `GH_PAT_REPO` and `RUNPOD_API_KEY` are now standard on every pod image, or whether Run 1's pod was a one-off misconfiguration and the next pod could just as easily boot without them again.
- Whether the self-termination path in `supervisor.sh` actually succeeds end-to-end here — this report is written before that happens, since writing DONE is what triggers it.
- Whether the same boot path holds under conditions this run didn't hit: a mid-run pod reclaim, a transient S3 outage, a CUDA OOM, an expiring token, or the smoke gate actually needing its one allowed auto-fix and retry.
- Whether a live Slack thread (once `SLACK_BOT_TOKEN` is set) actually delivers and receives correctly — nothing here exercised the receive path, only its absence.

If the next spike boots on a pod missing either credential again, the conclusion "the boot path works" should be treated as pod-specific luck, not a property of the image.

## 4. Decisions defaulted without the researcher

- **Skipped the live consultation session** `CLAUDE.md`'s general closing workflow calls for before writing the report. `SPEC.md` section 4 pre-registers an objective, pass/fail interpretation — nothing to discuss — and no transport exists yet that could hold that conversation anyway (`SLACK_BOT_TOKEN` unset). Posted this default via the real `notify` function (logged, delivery skipped) before proceeding. Run 1 made the same call for the same reason.
- **Skipped posting to Linear** in the closing workflow: no Linear API token or CLI (`gh` is also absent) is present in this environment. Noted here instead.
- **Did not open a draft PR.** This repo's own `README.md` states it exists only to test the pod boot path and is not the real experiments repo (`agencyenterprise/aiaf-fellowship-p2-selfie` is). Confirmed via the GitHub API that `aiaf-spike-testbed` has exactly one branch, `exp/RES-000-boot-smoke`, which is also its own default branch — there is no separate base branch to open a PR against here. `CLAUDE.md`'s closing workflow assumes a normal spikes repo; this throwaway one doesn't have the branch structure that step needs.

## 5. Cost

Well under $1 across both runs combined: a few seconds of GPU time each on a $0.59/hr pod (`RUNPOD_DC_ID=US-PA-1`), plus negligible S3 transfer for a ~1.4 KB checkpoint and a one-line log each time. No projection was needed before running — the entire arm was known in advance to be a few seconds of compute, far under the $25 threshold where `CLAUDE.md` asks for even a passing mention.

## 6. Parking lot

Nothing added to `SPEC.md` — a boot-path smoke test with a single pre-specified noop arm didn't surface a research idea worth parking. Two operational items worth someone's attention, recorded here rather than in `SPEC.md` since they aren't experimental arms:
- Confirm whether `GH_PAT_REPO` and `RUNPOD_API_KEY` are now baked into the standard pod image/boot config, or whether Run 1 was a one-off. If pods can still boot without either, the failure mode Run 1 hit (stranded code, indefinitely billing pod) can recur silently.
- Wire up `SLACK_BOT_TOKEN` for `$SLACK_THREAD` — the transport code is real and ready to receive, it just has no credential yet.

## 7. Recommended next step

Set `SLACK_BOT_TOKEN` and re-run this exact spike (or a near-identical one) to confirm the notify/block/default loop behaves correctly with a live channel — including a deliberate ASK to confirm a researcher's reply is picked up inside the 10-minute window. Separately, verify the credential/API-key fix that made Run 2 succeed is part of the standard pod image rather than incidental to this specific pod, since Run 1 shows what happens when it's missing. Once both are confirmed, this throwaway repo can be deleted per its own README.